In [1]:
import pandas as pd
import numpy as np
import pyodbc
import warnings
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.min_rows', 40)
pd.set_option('display.width', 200)

DSN = 'Redshift_prod_new'


def run_sql(query, connection=None):
    if connection is None:
        with pyodbc.connect(f'DSN={DSN}') as conn:
            warnings.filterwarnings('ignore', category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings('default', category=UserWarning)
            return df
    else:
        warnings.filterwarnings('ignore', category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings('default', category=UserWarning)
        return df


print(f'Imports loaded, DSN={DSN}')

Imports loaded, DSN=Redshift_prod_new


## Step 1: Introspect `sandbox.rds_rec_model_originations`
Pull all column names and types from the information schema to understand what vehicle-level data is available.

In [2]:
introspect_query = """
SELECT column_name,
       data_type,
       ordinal_position
FROM information_schema.columns
WHERE table_schema = 'sandbox'
  AND table_name   = 'rds_rec_model_originations'
ORDER BY ordinal_position
"""

schema_df = run_sql(introspect_query)
print(f'{len(schema_df)} columns in sandbox.rds_rec_model_originations\n')
display(schema_df)

43 columns in sandbox.rds_rec_model_originations



,column_name,data_type,ordinal_position
0,account_number,bigint,1
1,con_date,date,2
2,lob,character varying,3
3,state_pb,character varying,4
4,driver_flag,integer,5
5,retired_flag,integer,6
6,military_flag,integer,7
7,job_category,character varying,8
8,trade_flag,numeric,9
9,vin,character varying,10


## Step 2: Fetch data with vehicle-related columns

After reviewing the schema above, adjust the column list below to include every
vehicle-descriptive column available (make, model year, fuel type, body style,
new/used, mileage, etc.).  The query also pulls the current standardised
depreciation rate so we can bucket contracts.

In [3]:
# ---------------------------------------------------------------------------
# After running the introspection cell above, paste the vehicle-related column
# names you found into this query.  The template below is a best-guess from the
# columns seen across the repo; uncomment / adjust as needed once you see the
# actual schema output.
# ---------------------------------------------------------------------------

vehicle_cols = [c for c in schema_df.column_name.tolist() if any(
    kw in c.lower() for kw in [
        'make', 'model', 'year', 'fuel', 'body', 'type', 'new_used',
        'mileage', 'odometer', 'vin', 'class', 'drivetrain', 'engine',
        'transmission', 'trim', 'ev', 'electric', 'hybrid', 'vehicle',
        'msrp', 'bb_value', 'bbvalue', 'sale', 'price',
    ]
)]

core_cols = [
    'account_number', 'con_date', 'lob',
    'pred_depr_rate_std', 'pred_depr_rate_std_current',
    'pred_depr_rate_raw_current',
]

all_cols = core_cols + [c for c in vehicle_cols if c not in core_cols]
col_list = ',\n       '.join(all_cols)

print(f'Vehicle-related columns found: {vehicle_cols}')
print(f'\nTotal columns to fetch: {len(all_cols)}')

data_query = f"""
SELECT {col_list}
FROM sandbox.rds_rec_model_originations
WHERE con_date >= '2022-01-01'
  AND con_date <  '2026-07-01'
  AND pred_depr_rate_std_current IS NOT NULL
"""

print(f'\nQuery:\n{data_query}')
df = run_sql(data_query)

df['con_date'] = pd.to_datetime(df['con_date'])
print(f'\nFetched {len(df):,} records')
print(f'Date range: {df.con_date.min().date()} to {df.con_date.max().date()}')
print(f'\npred_depr_rate_std_current summary:')
display(df['pred_depr_rate_std_current'].describe().round(4))

Vehicle-related columns found: ['vin', 'vin10', 'veh_year', 'veh_make', 'veh_make_grp', 'veh_model', 'mileage_orig', 'mileage_orig_capped', 'bb_value', 'kmx_sale_price', 'veh_class_raw', 'veh_class_grp', 'veh_trim', 'veh_fuel_raw', 'veh_fuel_grp', 'vin10_mapped_flag', 'mileage_auc']

Total columns to fetch: 23

Query:

SELECT account_number,
       con_date,
       lob,
       pred_depr_rate_std,
       pred_depr_rate_std_current,
       pred_depr_rate_raw_current,
       vin,
       vin10,
       veh_year,
       veh_make,
       veh_make_grp,
       veh_model,
       mileage_orig,
       mileage_orig_capped,
       bb_value,
       kmx_sale_price,
       veh_class_raw,
       veh_class_grp,
       veh_trim,
       veh_fuel_raw,
       veh_fuel_grp,
       vin10_mapped_flag,
       mileage_auc
FROM sandbox.rds_rec_model_originations
WHERE con_date >= '2022-01-01'
  AND con_date <  '2026-07-01'
  AND pred_depr_rate_std_current IS NOT NULL


Fetched 571,454 records
Date range: 2022-01-0

count    571454.0000
mean          0.2062
std           0.0476
min           0.0193
25%           0.1660
50%           0.2216
75%           0.2370
max           0.3605
Name: pred_depr_rate_std_current, dtype: float64

## Step 3: Bucket into Low (<20%) and High (>25%) depreciation

In [4]:
LOW_THRESHOLD  = 0.20
HIGH_THRESHOLD = 0.25

df['depr_bucket'] = np.where(
    df['pred_depr_rate_std_current'] < LOW_THRESHOLD, 'Low (<20%)',
    np.where(
        df['pred_depr_rate_std_current'] > HIGH_THRESHOLD, 'High (>25%)',
        'Middle (20-25%)'
    )
)

bucket_counts = df['depr_bucket'].value_counts()
bucket_pcts   = df['depr_bucket'].value_counts(normalize=True).mul(100).round(2)

summary = pd.DataFrame({'count': bucket_counts, 'pct': bucket_pcts})
print('Depreciation bucket distribution:')
display(summary)

low_df  = df[df['depr_bucket'] == 'Low (<20%)'].copy()
high_df = df[df['depr_bucket'] == 'High (>25%)'].copy()

print(f'\nLow  depreciation contracts: {len(low_df):,}')
print(f'High depreciation contracts: {len(high_df):,}')

Depreciation bucket distribution:


,count,pct
depr_bucket,,
Middle (20-25%),304350,53.26
Low (<20%),180935,31.66
High (>25%),86169,15.08



Low  depreciation contracts: 180,935
High depreciation contracts: 86,169


## Step 4: Profile extreme buckets by vehicle attributes

The cells below automatically use whichever vehicle columns were found during
introspection.  Each categorical column is summarised as a frequency table within
each bucket; numerical columns get descriptive statistics.

In [5]:
extreme_df = df[df['depr_bucket'].isin(['Low (<20%)', 'High (>25%)'])].copy()

profile_cols = [c for c in vehicle_cols if c in extreme_df.columns]

for col in profile_cols:
    print('=' * 80)
    print(f'  {col}')
    print('=' * 80)

    if extreme_df[col].dtype in ('object', 'category') or extreme_df[col].nunique() <= 30:
        tbl = (
            extreme_df
            .groupby(['depr_bucket', col])
            .size()
            .unstack(fill_value=0)
            .T
        )
        tbl['total'] = tbl.sum(axis=1)
        tbl = tbl.sort_values('total', ascending=False)
        for bucket in ['Low (<20%)', 'High (>25%)']:
            if bucket in tbl.columns:
                tbl[f'{bucket} %'] = (tbl[bucket] / tbl[bucket].sum() * 100).round(2)
        display(tbl.head(30))
    else:
        display(
            extreme_df
            .groupby('depr_bucket')[col]
            .describe()
            .round(4)
        )
    print()

  vin


,count,unique,top,freq
depr_bucket,,,,
High (>25%),86169,85202,KL8CL6S07FC760866,5
Low (<20%),180935,179349,5YFBURHE4EP080689,3



  vin10


,count,unique,top,freq
depr_bucket,,,,
High (>25%),86169,35039,5YJ3E1EA6P,59
Low (<20%),180935,56137,5YFEPMAE9N,412



  veh_year


...

depr_bucket,High (>25%),Low (<20%),total,Low (<20%) %,High (>25%) %
veh_fuel_grp,,,,,
Gas,75104,163382,238486,90.30,87.16
EV,8894,584,9478,0.32,10.32
Hybrid,1026,7416,8442,4.10,1.19
Diesel,580,5891,6471,3.26,0.67
Flex,564,3611,4175,2.00,0.65
diesel,1,51,52,0.03,0.00



  vin10_mapped_flag


depr_bucket,High (>25%),Low (<20%),total,Low (<20%) %,High (>25%) %
vin10_mapped_flag,,,,,
0,75519,159352,234871,88.07,87.64
1,10650,21583,32233,11.93,12.36



  mileage_auc


,count,mean,std,min,25%,50%,75%,max
depr_bucket,,,,,,,,
High (>25%),16909.0,104047.9485,41291.8982,1.0,74855.00,100569.0,130546.00,339349.0
Low (<20%),29562.0,108605.2164,53774.5861,1.0,70835.75,102752.0,140434.25,777777.0


## Step 5: Combined comparison table

A single side-by-side table showing how each vehicle attribute distributes across
the two extreme depreciation buckets.

In [6]:
cat_cols = [c for c in profile_cols
            if extreme_df[c].dtype in ('object', 'category') or extreme_df[c].nunique() <= 30]

rows = []
for col in cat_cols:
    for bucket_label in ['Low (<20%)', 'High (>25%)']:
        subset = extreme_df[extreme_df['depr_bucket'] == bucket_label]
        top = subset[col].value_counts(normalize=True).head(10)
        for val, pct in top.items():
            rows.append({
                'attribute': col,
                'value': val,
                'bucket': bucket_label,
                'pct': round(pct * 100, 2),
                'count': int(subset[col].eq(val).sum()),
            })

comparison_df = pd.DataFrame(rows)

if not comparison_df.empty:
    pivot = comparison_df.pivot_table(
        index=['attribute', 'value'],
        columns='bucket',
        values='pct',
        fill_value=0,
    ).reset_index()

    count_pivot = comparison_df.pivot_table(
        index=['attribute', 'value'],
        columns='bucket',
        values='count',
        fill_value=0,
    ).reset_index()

    combined = pivot.copy()
    for bucket_label in ['Low (<20%)', 'High (>25%)']:
        if bucket_label in count_pivot.columns:
            combined[f'{bucket_label} n'] = count_pivot[bucket_label]

    print('Top values per attribute by depreciation bucket (% within bucket):\n')
    display(combined)
else:
    print('No categorical vehicle columns found for comparison.')

Top values per attribute by depreciation bucket (% within bucket):



bucket,attribute,value,High (>25%),Low (<20%),Low (<20%) n,High (>25%) n
0,veh_class_grp,Car,45.17,22.00,39801.0,38924.0
1,veh_class_grp,Compact,3.01,24.19,43777.0,2594.0
2,veh_class_grp,Minivan,0.23,1.62,2929.0,198.0
3,veh_class_grp,SUV Large,32.07,6.52,11794.0,27637.0
4,veh_class_grp,SUV Small,19.50,10.99,19879.0,16804.0
5,veh_class_grp,Sporty,0.01,9.05,16381.0,5.0
6,veh_class_grp,Truck,0.00,24.82,44908.0,4.0
7,veh_class_grp,Work Van,0.00,0.81,1466.0,3.0
8,veh_class_raw,Full-Size Car,0.98,0.00,0.0,845.0
9,veh_class_raw,Large Crossover/SUV,16.23,4.71,8521.0,13986.0


## Step 6: LOB breakdown within each extreme bucket

In [7]:
lob_breakdown = (
    extreme_df
    .groupby(['depr_bucket', 'lob'])
    .agg(
        n=('account_number', 'count'),
        mean_depr=('pred_depr_rate_std_current', 'mean'),
    )
    .reset_index()
)

lob_breakdown['pct_within_bucket'] = (
    lob_breakdown.groupby('depr_bucket')['n']
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)
lob_breakdown['mean_depr'] = lob_breakdown['mean_depr'].round(4)

print('LOB distribution within each extreme depreciation bucket:\n')
display(
    lob_breakdown.pivot_table(
        index='lob',
        columns='depr_bucket',
        values=['n', 'pct_within_bucket', 'mean_depr'],
    ).fillna(0)
)

LOB distribution within each extreme depreciation bucket:



mean_depr                      n            pct_within_bucket           
depr_bucket High (>25%) Low (<20%) High (>25%) Low (<20%)       High (>25%) Low (<20%)
lob                                                                                   
AN               0.2634     0.1437      6021.0    14571.0              6.99       8.05
ENT              0.2742     0.1339      4715.0    18787.0              5.47      10.38
FLD              0.2642     0.1436      4305.0     7880.0              5.00       4.36
FRN              0.2642     0.1464     14627.0    34703.0             16.97      19.18
KMX              0.2622     0.1452     46366.0    69954.0             53.81      38.66
SFS              0.2614     0.1561      2993.0     9972.0              3.47       5.51
STG              0.2631     0.1421      7142.0    25068.0              8.29      13.85